## Proofreading Student Essays

### Motivation
This competition aims to provide "overtaxed teachers" with the tools needed to provide timely feedback to students, particularly those in underserved communities. While working on automating the scoring of student essays, we discovered an adjacent opportunity to automate the proofreading of these essays. When paired with automated scoring, automated proofreading may provide students with free or low-cost access to instant writing feedback and training.

### Approach
We, through fine-tuning, coaxed an open-weight LLM, Gemma 2B, into proofreading a student essay while preserving as much of its original content as possible. The resulting model corrects grammatical errors and may suggest edits to improve clarity.

### Challenges
* Every LLM we tried possessed a strong tendency to fully rewrite an essay regardless of whether it has been prompted to correct "grammatical issues only". Prompt adherence required fine-tuning through use of example revisions containing more appropriate levels of rewriting. The dataset for fine-tuning was constructed as follows:
  * We used a larger LLM, Gemma 7B, to generate multiple revisions of sample essays.
  * We chose revisions that edited a target fraction of an essay. Lower-scoring essays had higher targets than higher-scoring essays. For example, we included in our dataset revisions that edited between 25% and 40% of an essay with a score of 1 but only included revisions that edited under 15% of an essay with a score of 6. 
  * We rejected revisions with too many or too few edits. Using "chosen" and "rejected" results, revisions were collated to form a dataset for ORPO training.
* Every LLM we tried struggled to include content from every sentence of the original essay. This issue relates to the previous issue identified but more specifically refers to a tendency for the LLM to "lose its place" as it revises an essay. To address this issue, essays are first pre-processed. Sentences of an essay are enumerated and listed on separate lines. Once a revision for each sentence is received, we fold the changes back into the orignal essay.
* A revision is most useful for writing feedback when changes are shown as inline edits to an essay. A library named `diff-match-patch` is used to extract "insertions" and "deletions" to enable a quick comparison of the original essay to the revised essay.

## Example Result

In [ ]:
from IPython.display import display, HTML

display(HTML("""
<html lang="en">
<head>
    <meta charset="utf-8">
    <style>
        section { white-space: pre-wrap; }
        ins { background:#e6ffe6; }
        del { background:#ffe6e6; }
        body:has(#Original:checked) ins { display: none; }
        body:has(#Original:checked) del { text-decoration: none; }
        body:has(#Revised:checked) del { display: none; }
    </style>
</head>
<body>
<form>
    <input type="radio" name="option" id="Diff" checked><label for="Diff">Diff</label>
    <input type="radio" name="option" id="Original"><label for="Original">Original</label>
    <input type="radio" name="option" id="Revised"><label for="Revised">Revised</label>
</form>
<hr/>

<section><del>First,</del>The face NASA found <del>i</del><ins>o</ins>n Mars is just a landform<ins>,</ins> and I can prove it. <del>O</del><ins>The face o</ins>n the picture <del>on</del><ins>from</ins> 1976 <del>it </del>act<del>a</del>u<ins>a</ins>lly looked like a human face, but a human face can<ins>no</ins>t be that big. <del>On 1998 t</del><ins>T</ins>he eyes and mouth started fading away like it was disap<ins>p</ins>earing<del>. On 2001, you can't even see the face anymor</del><ins> on the 1998 picture. By 2001, the face was no longer visibl</ins>e.

<del>Next, </del>It looked like it cracked, a<del>lso</del><ins>nd</ins> scientists <del>'figur</del><ins>initially believ</ins>ed it was just another Martian <del>M</del><ins>m</ins>esa, comm<del>an enough</del><ins>on</ins> around Cydonia.<del>' But, also</del><ins> However,</ins> scientists <ins>also </ins>believed the face was an alien artifact<ins>,</ins> but<del>,</del> they can<del>'</del><ins>no</ins>t prove th<del>at it is just what they believe</del><ins>is</ins>.

Then,<del>O</del><ins> o</ins>n April 5, 1998<del> it said that</del><ins>,</ins> Mars Global Surveyor flew over to Cydonia to see if it was real. When they <del>got there</del><ins>arrived,</ins> they took some pictures of it for <ins>the </ins>JPL web<del> </del>site, but it revealed that it was just a natural landform and <del>it wasn'</del><ins>no</ins>t a<ins>n</ins> alien monument.

Finally<del> </del>,<del> just to make sure</del> on April 8, 2001<del> it said in</del><ins>,</ins> the article <ins>stated </ins>that "Mars Global Surveyor drew close enough for a second look.<del> "</del>"<ins> </ins>They rolled the spacecraft 25 degrees to center the Face in the field of view,<del>"</del> said Garvin. <del>What t</del><ins>T</ins>he picture act<del>a</del>u<ins>a</ins>lly shows <del>is </del>the Martian equivalent of a butte or mesa-<ins>a</ins> landform common around the American West.

In conclusion, there was no alien or face <del>i</del><ins>o</ins>n Mars. It was just a landform <del>making</del><ins>that made</ins> it look like there was a human face. This is my essay <del>telling it</del><ins>explaining that the face</ins> was just a landform.</section>
<hr/>
</body>
</html>
"""))

## Setup

In [ ]:
%pip install nltk torch diff-match-patch peft transformers

## Helpers

In [ ]:
"""
Collection of proofreading-related utilities.
"""

import re
from functools import cached_property
from typing import Optional

import nltk
import torch
from diff_match_patch import diff_match_patch
from nltk.tokenize import sent_tokenize
from peft.peft_model import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


class Revision:
    """
    Object representing a revision of a text.
    """

    def __init__(self, llm_inputs: list[str], llm_outputs: list[str], diffs: Optional[list[tuple[int, str]]] = None):
        """
        A revision is internally represented as a list of diffs.
        """
        # Inputs to and outputs from the LLM are stored to enable reinforcement learning.
        self.llm_inputs = llm_inputs
        self.llm_outputs = llm_outputs
        self.diffs = diffs or self._create_diffs()

    def as_json(self) -> dict:
        """
        Returns the revision as a dictionary.
        """
        return {
            "llm_inputs": self.llm_inputs,
            "llm_outputs": self.llm_outputs,
            "diffs": self.diffs,
        }

    @classmethod
    def from_json(cls, data: dict):
        """
        Creates a Revision object from a dictionary.
        """
        return cls(data["llm_inputs"], data["llm_outputs"], data["diffs"])

    def as_original_text(self) -> str:
        """
        Returns a cleaned-up version of the original full text that was used to generate revisions.
        """
        cleaned_text = ""
        for op, data in self.diffs:
            if op in (diff_match_patch.DIFF_EQUAL, diff_match_patch.DIFF_DELETE):
                cleaned_text += data
        return cleaned_text

    def as_revised_text(self) -> str:
        """
        Returns the revised full text.
        """
        revised_text = ""
        for op, data in self.diffs:
            if op in (diff_match_patch.DIFF_EQUAL, diff_match_patch.DIFF_INSERT):
                revised_text += data
        return revised_text

    def as_html_fragment(self) -> str:
        """
        Returns the revision as an HTML string without a parent element.
        """
        html_fragments = []
        for op, data in self.diffs:
            text = data.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br/><br/>")
            if op == diff_match_patch.DIFF_INSERT:
                html_fragments.append("<ins>%s</ins>" % text)
            elif op == diff_match_patch.DIFF_DELETE:
                html_fragments.append("<del>%s</del>" % text)
            elif op == diff_match_patch.DIFF_EQUAL:
                html_fragments.append("%s" % text)
        html = "".join(html_fragments)
        return html

    def as_html_document(self, score: Optional[int] = None) -> str:
        """
        Returns the revision as HTML content with a parent <html> element.
        """
        html_content = """
        <html lang="en">
        <head>
            <meta charset="utf-8">
            <style>
                section { white-space: pre-wrap; }
                ins { background:#e6ffe6; }
                del { background:#ffe6e6; }
                body:has(#Original:checked) ins { display: none; }
                body:has(#Original:checked) del { text-decoration: none; }
                body:has(#Revised:checked) del { display: none; }
            </style>
        </head>
        <body>
        {{ score_section }}
        {{ essay_section }}
        <hr/>
        <form>
            <input type="radio" name="option" id="Diff" checked><label for="Diff">Diff</label>
            <input type="radio" name="option" id="Original"><label for="Original">Original</label>
            <input type="radio" name="option" id="Revised"><label for="Revised">Revised</label>
        </form>
        </body>
        </html>
        """
        score_section = f"<section>Score: { score }</section><hr/>" if score else ""
        essay_section = f"<section>{ self.as_html_fragment() }</section>"
        return html_content.replace("{{ score_section }}", score_section).replace("{{ essay_section }}", essay_section)

    @cached_property
    def diff_size(self) -> int:
        """
        Returns the number of characters in the diff.
        """
        # For now, just use the size of the larger of the two types of diffs.
        del_size = sum([len(data) for (op, data) in self.diffs if op == -1])
        ins_size = sum([len(data) for (op, data) in self.diffs if op == 1])
        diff_size = max(del_size, ins_size)
        return diff_size

    @cached_property
    def percent_change(self) -> float:
        """
        Returns the percentage of text changed in the revision.
        """
        original_text = self.as_original_text()
        return min(self.diff_size / len(original_text) * 100, 100.0)

    def _create_diffs(self) -> list[tuple[int, str]]:
        """
        Creates a list of diffs between the original and revised text.
        """
        all_diffs = []

        # Expects each sentence to be on a new line that starts with a number in brackets. (e.g.: [1] Sentence 1)
        for llm_input, llm_output in zip(self.llm_inputs, self.llm_outputs):

            # Remove empty lines from revised essays
            modified_text_lines = []
            for line in llm_output.split("\n"):
                stripped_line = line.strip()
                if stripped_line == "":
                    continue
                if re.sub(r"^\[(\d+)\]$", "", stripped_line) == "":
                    continue
                modified_text_lines.append(line)
            modified_text = "\n".join(modified_text_lines)

            # Replace smart quotes with ascii quotes
            original_text = llm_input.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
            modified_text = modified_text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")

            # Replace different variations of hypens with an ascii hypen
            original_text = re.sub(r"[-‐‑‒–—―−]", "-", original_text)
            modified_text = re.sub(r"[-‐‑‒–—―−]", "-", modified_text)

            # Regex: \[\d+\] -> __________
            sentence_start_token = "_" * 128
            # Strip space at the start of the essay
            original_text = re.sub(r"^ ?\[(\d+)\] ", sentence_start_token, original_text)
            modified_text = re.sub(r"^ ?\[(\d+)\] ", sentence_start_token, modified_text)
            # Strip spaces at the beginning of paragraphs
            original_text = re.sub(r"\n\n ?\[(\d+)\] ", "\n\n" + sentence_start_token, original_text, re.MULTILINE)
            # Include space before the start of sequential sentence in a paragraph.
            original_text = re.sub(r"\[(\d+)\] ", " " + sentence_start_token, original_text)
            modified_text = re.sub(r"\[(\d+)\] ", " " + sentence_start_token, modified_text)

            # Double all spaces to increase the cost of creating a diff spanning multiple words.
            original_text = re.sub(r" ", "  ", original_text)
            modified_text = re.sub(r" ", "  ", modified_text)

            # Create diffs
            dmp = diff_match_patch()
            diffs = dmp.diff_main(original_text, modified_text, checklines=False)

            # Filter out the sentence start token
            diffs = [
                (op, data.replace(sentence_start_token, "")) for (op, data) in diffs if data != sentence_start_token
            ]

            # Remove doubled spaces.
            diffs = [(op, re.sub(r"  ", " ", data)) for (op, data) in diffs]

            # Add back in paragraph breaks, remove newlines after every sentence, and remove whitespace before the start of paragraphs.
            updated_diffs = []
            paragraph_start = True
            for op, data in diffs:
                if op == dmp.DIFF_DELETE and data == "\n":
                    updated_diffs.append((dmp.DIFF_EQUAL, "\n"))
                    paragraph_start = True
                else:
                    data = data.replace("\n", "")
                    if paragraph_start:
                        data = re.sub(r"^\s+", "", data)
                    if data != "":
                        updated_diffs.append((op, data))
                    paragraph_start = False
            diffs = updated_diffs

            # Reduce the number of edits by eliminating semantically trivial equalities.
            dmp.diff_cleanupSemantic(diffs)

            # If we will be appending more diffs from another paragraph, add a trailing newline.
            if llm_input is not self.llm_inputs[-1]:
                diffs.append((dmp.DIFF_EQUAL, "\n"))

            all_diffs.extend(diffs)

        return all_diffs


class Proofreader:
    """
    Uses Gemma to proofread an essay.
    """

    # A document needs to be split into multiple chunks if too large.
    TARGET_CHUNK_SIZE = 550

    def __init__(self, base_model_name: str, checkpoint_path: str, temperature: float = 0):
        """
        Loads LLM for inference.
        """
        # Downloads required configuration for nltk.
        nltk.download("punkt")

        # Load model
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            device_map="auto",
        )
        model = PeftModel.from_pretrained(base_model, checkpoint_path)
        model.merge_and_unload()

        # Compile model
        model.generation_config.max_length = 2048
        if temperature > 0:
            model.generation_config.temperature = temperature
            model.generation_config.do_sample = True
        else:
            model.generation_config.do_sample = False
        self.model = model
        self.tokenizer = AutoTokenizer.from_pretrained(checkpoint_path, padding_side="left")

    def proofread_essays(self, full_texts: list[str]) -> list[Revision]:
        """
        Returns a revision of the given text.
        """
        # Chunks of text to decode.
        llm_inputs = []

        # Maps input index to revision index.
        index_map = []

        # Create chunks.
        for i, full_text in enumerate(full_texts):
            cleaned_text = self._preprocess_text(full_text)
            chunks = self._create_chunks(cleaned_text)
            llm_inputs.extend(chunks)
            index_map.extend([i] * len(chunks))

        # Create prompts.
        prompts = [self.create_proofreading_prompt(llm_input) for llm_input in llm_inputs]

        # Add response prefixes to coax desired behavior from the model.
        prefixes = [llm_input.split(" ", 1)[0] for llm_input in llm_inputs]
        prompts_with_prefix = [prompt + prefix for prompt, prefix in zip(prompts, prefixes)]

        # Tokenize prompts.
        tokenized_prompts = self.tokenizer(prompts_with_prefix, padding=True, return_tensors="pt").to(self.model.device)
        tokenized_prefixes = self.tokenizer(prefixes, padding=False, add_special_tokens=False)

        # Run model.
        output_ids = self.model.generate(**tokenized_prompts)

        # Extract responses.
        prompt_lengths = [
            len(ids) - len(prefix_ids)
            for ids, prefix_ids in zip(tokenized_prompts["input_ids"], tokenized_prefixes["input_ids"])  # type: ignore
        ]
        generated_ids = [ids[prompt_length:] for prompt_length, ids in zip(prompt_lengths, output_ids)]
        llm_outputs = self.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

        # Create revisions
        revisions = []
        for i in range(len(full_texts)):
            revision_inputs = [llm_input for index, llm_input in zip(index_map, llm_inputs) if index == i]
            revision_outputs = [llm_output for index, llm_output in zip(index_map, llm_outputs) if index == i]
            revision = Revision(revision_inputs, revision_outputs)
            revisions.append(revision)

        return revisions

    def create_proofreading_prompt(self, text: str, instructions: Optional[list[str]] = None) -> str:
        """
        Creates a Gemma-specific proofreading prompt for the given text.
        """
        if instructions is None:
            # Default instructions if none are provided.
            instructions = [
                "Please proofread the following essay.",
                "It is very important that you preserve as much of the original text as possible.",
                "You may make small and infrequent edits that correct grammatical issues only.",
                "Preserve the location of the numbers in the brackets.",
                "If a sentence has no grammatical errors, repeat the sentence verbatim.",
                "Do not explain your edits or add notes.",
            ]

        prompt = f"""<start_of_turn>user
            {" ".join(instructions)}

            Essay:
            {text}
            <end_of_turn>
            <start_of_turn>model
            Revised Essay:
            """

        # Trim space before each new line.
        prompt = re.sub(r"\n\s*", "\n", prompt)
        return prompt

    def _create_chunks(self, full_text: str) -> list[str]:
        """
        Splits the full text into chunks and starts each sentence on a new line that begins with a number in brackets.
        """
        chunks = []
        input_text = ""
        count = 0
        for paragraph in map(lambda p: p.strip(), re.split("\n\n|\n", full_text)):
            p_word_count = len(paragraph.split())
            # Each sentence should be on a new line that start with a number in brackets.
            sentences = map(lambda s: s.strip(), sent_tokenize(paragraph))
            for s in sentences:
                # If essay has long paragraphs, allow intra-paragraph breaks.
                if p_word_count > self.TARGET_CHUNK_SIZE / 3:
                    if len(input_text.split()) > self.TARGET_CHUNK_SIZE:
                        chunks.append(input_text.strip())
                        input_text = ""
                count += 1
                input_text += f"[{count}] {s}\n"
            input_text += "\n"
            # Start a new chunk after a certain length
            if len(input_text.split()) > self.TARGET_CHUNK_SIZE:
                chunks.append(input_text.strip())
                input_text = ""

        if input_text:
            chunks.append(input_text.strip())

        return chunks

    def _preprocess_text(self, text) -> str:
        """
        Clean up input text before processing.
        """
        # The documents seem to be OCRs. Replace double single quotes with double quotes.
        text = text.replace("''", '"')
        return text


## Code

In [ ]:
import pandas

BASE_MODEL_NAME = "/kaggle/input/gemma/transformers/2b-it/3"
CHECKPOINT_PATH = "/kaggle/input/proofreading/transformers/proofreader/1"

# Create proofreader.
proofreader = Proofreader(BASE_MODEL_NAME, CHECKPOINT_PATH)

In [ ]:
# Load test essays.
test_essays_path = "/kaggle/input/learning-agency-lab-automated-essay-scoring-2/test.csv"
test_essays = pandas.read_csv(test_essays_path)

# Get the first three essays.
full_texts = test_essays["full_text"].tolist()[:3]

# Batch process full texts.
revisions = proofreader.proofread_essays(full_texts)

# Display the revisions.
for i, revision in enumerate(revisions):
    print(f'\n{test_essays["essay_id"][i]}')
    display(HTML(revision.as_html_fragment()))